In [11]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
    

CUDA available: True
GPU name: NVIDIA L4


In [12]:
x = torch.rand(3, 3).to("cuda")
print(x.device)

cuda:0


In [13]:
import boto3

session = boto3.Session()
print(session.get_credentials())

In [14]:
import boto3

sts = boto3.client("sts")
print(sts.get_caller_identity())

{'UserId': 'AROAUJXCMC4VF2CLCZ7Q5:SageMaker', 'Account': '295753750314', 'Arn': 'arn:aws:sts::295753750314:assumed-role/AmazonSageMaker-ExecutionRole-20260423T015477/SageMaker', 'ResponseMetadata': {'RequestId': '8d7dca16-aba0-40be-ba24-8a3e4c765706', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amzn-requestid': '8d7dca16-aba0-40be-ba24-8a3e4c765706', 'x-amz-sts-extended-request-id': 'MTp1cy1lYXN0LTI6UzoxNzc3MDE3NTM1MDk4OlI6Z01sdHIwZWQ=', 'content-type': 'text/xml', 'content-length': '470', 'date': 'Fri, 24 Apr 2026 07:58:55 GMT'}, 'RetryAttempts': 0}}


In [39]:
import os
import zipfile
import boto3
import torch
import torch.nn as nn
from torch import optim
from torchvision.models import vit_b_32
from torchvision import transforms, datasets
from torch.utils.data import DataLoader, random_split

def create_vit_model(num_encoder_layers=2, num_classes=2):
    model = vit_b_32(weights=None)
    model.encoder.layers = nn.Sequential(
        *[model.encoder.layers[i] for i in range(num_encoder_layers)]
    )
    model.heads.head = nn.Linear(model.heads.head.in_features, num_classes)
    return model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Using device: cuda
GPU: NVIDIA L4


In [40]:
bucket = "vit-stock-data-ikenna14"

s3_train_up = "vit-data/train/up/train_up.zip"
s3_train_down = "vit-data/train/down/train_down.zip"

BASE_DIR = "/mnt/sagemaker-nvme/vit_data"

train_up_path = f"{BASE_DIR}/train_up.zip"
train_down_path = f"{BASE_DIR}/train_down.zip"

os.makedirs(f"{BASE_DIR}/train/up", exist_ok=True)
os.makedirs(f"{BASE_DIR}/train/down", exist_ok=True)

In [41]:
# Block 3 — download zips from S3

s3 = boto3.client("s3") 

if not os.path.exists(train_up_path):
    s3.download_file(
        bucket,
        "vit-data/train/up/Train_up.zip",
        train_up_path
    )

if not os.path.exists(train_down_path):
    s3.download_file(
        bucket,
        "vit-data/train/down/Train_down.zip",
        train_down_path
    )

In [42]:
os.makedirs("data/train/up", exist_ok=True)
os.makedirs("data/train/down", exist_ok=True)

with zipfile.ZipFile(train_up_path, 'r') as z:
    z.extractall(f"{BASE_DIR}/train/up/up")

with zipfile.ZipFile(train_down_path, 'r') as z:
    z.extractall(f"{BASE_DIR}/train/down/down")

print("Unzip complete")

Unzip complete


In [43]:
import os

up_path = f"{BASE_DIR}/train/up/up"
down_path = f"{BASE_DIR}/train/down/down"

print("Up images:", len(os.listdir(up_path)))
print("Down images:", len(os.listdir(down_path)))

Up images: 758757
Down images: 715305


In [46]:
print(os.listdir(f"{BASE_DIR}/train/up"))
print(os.listdir(f"{BASE_DIR}/train/down"))

['up']
['down']


In [47]:
import shutil
import os

BASE = f"{BASE_DIR}/train"

shutil.move(f"{BASE}/up/up", f"{BASE}/up_tmp")
shutil.move(f"{BASE}/down/down", f"{BASE}/down_tmp")

shutil.rmtree(f"{BASE}/up")
shutil.rmtree(f"{BASE}/down")

os.rename(f"{BASE}/up_tmp", f"{BASE}/up")
os.rename(f"{BASE}/down_tmp", f"{BASE}/down")

print("fixed")

fixed


In [48]:
print(os.listdir(f"{BASE_DIR}/train/up")[:5])
print(os.listdir(f"{BASE_DIR}/train/down")[:5])

['10001_19930112.png', '10001_19930113.png', '10001_19930114.png', '10001_19930115.png', '10001_19930118.png']
['10001_19930104.png', '10001_19930105.png', '10001_19930106.png', '10001_19930107.png', '10001_19930108.png']


In [54]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

full_dataset = datasets.ImageFolder(
    root=f"{BASE_DIR}/train",
    transform=transform
)





In [55]:
from torch.utils.data import random_split

full_size = len(full_dataset)

train_size = int(0.7 * full_size)
val_size = full_size - train_size

train_dataset, val_dataset = random_split(
    full_dataset,
    [train_size, val_size]
)

In [58]:
batch_size = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Classes:", full_dataset.classes)
print("Total samples:", len(train_dataset))

Classes: ['down', 'up']
Total samples: 2063683


In [62]:

model = create_vit_model(num_encoder_layers=2, num_classes=2)
model = model.to(device)

print("Model ready on:", device)

Model ready on: cuda


In [60]:
import torch.nn as nn
import torch.optim as optim

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)

print("Loss and optimizer ready")

Loss and optimizer ready


In [67]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for batch_idx, (images, labels) in enumerate(loader):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        preds = torch.argmax(outputs, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        if batch_idx % 1000 == 0:
            print(f"Train Batch {batch_idx} | Loss: {loss.item():.4f}")

    epoch_loss = running_loss / total
    epoch_acc = correct / total

    return epoch_loss, epoch_acc


@torch.no_grad()
def validate_one_epoch(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        outputs = model(images)
        loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)

        preds = torch.argmax(outputs, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc = correct / total

    return epoch_loss, epoch_acc

In [68]:
num_epochs = 10
best_val_acc = 0.0

save_path = "best_vit_model.pth"
checkpoint_path = "checkpoint.pth"

start_epoch = 0

In [69]:
import os

if os.path.exists(checkpoint_path):
    print("Loading checkpoint...")
    checkpoint = torch.load(checkpoint_path)

    model.load_state_dict(checkpoint['model_state'])
    optimizer.load_state_dict(checkpoint['optimizer_state'])

    start_epoch = checkpoint['epoch'] + 1
    best_val_acc = checkpoint['best_val_acc']

    print(f"Resuming from epoch {start_epoch}")

In [ ]:
for epoch in range(start_epoch, num_epochs):

    train_loss, train_acc = train_one_epoch(
        model, train_loader, criterion, optimizer, device
    )

    val_loss, val_acc = validate_one_epoch(
        model, val_loader, criterion, device
    )

    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}"
    )

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), save_path)
        print(f"Saved best model to {save_path}")

    # Save checkpoint every epoch
    torch.save({
        'epoch': epoch,
        'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'best_val_acc': best_val_acc
    }, checkpoint_path)

print(f"Best validation accuracy: {best_val_acc:.4f}")

Train Batch 0 | Loss: 0.6801
Train Batch 1000 | Loss: 0.6242
Train Batch 2000 | Loss: 0.7011
Train Batch 3000 | Loss: 0.7637
Train Batch 4000 | Loss: 0.7654
Train Batch 5000 | Loss: 0.7659
Train Batch 6000 | Loss: 0.7161
Train Batch 7000 | Loss: 0.7241
Train Batch 8000 | Loss: 0.7761
Train Batch 9000 | Loss: 0.7671
Train Batch 10000 | Loss: 0.6924
Train Batch 11000 | Loss: 0.6037
